In [ ]:
import json
import numpy as np
import pandas as pd
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import DATA_PATH, FULL_RUN, OXCGRT_COLMAP, LOCATION_NAME_MAP
from emu_renewal.utils import get_analysis_paths
from emu_renewal.plotting import plot_param_map, plot_best_policy

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
policy_codes = OXCGRT_COLMAP["custom"]

records = []
for iso3, analyses in analysis_paths.items():
    analysis_path = analyses["oxcgrt_independent"]
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["ts_weights"].median(dim=("chain", "draw"))
    best_policy = int(medians.argmax().item())
    row = {
        "ISO_A3": iso3,
        "best_policy": best_policy,
        "best_name": policy_codes[int(best_policy)]
    }
    for k, v in enumerate(medians):
        row[f"pol_{k}"] = float(v)
    records.append(row)
data_df = pd.DataFrame.from_records(records)

world = world.merge(data_df, on="ISO_A3", how="left")
missing = world[world["best_policy"].isna()]

In [ ]:
plot_best_policy(world, missing, None)

In [ ]:
for a, code in enumerate(policy_codes):
    policy = LOCATION_NAME_MAP[code]
    plot_param_map(world, f"pol_{a}", None, excluded=None, title=policy)